# Evaluador de calidad de datos abiertos
**Proyecto:** Desarrollo de un prototipo de software para la evaluación automatizada de la calidad de datos abiertos: completitud, unicidad y consistencia<br>
**Autora:** Ana Yibe Zamora Rivera · Semillero Sedetes · UNIMINUTO Virtual<br>
**Versión:** v1.2 — Partes 1 y 2 de 4: obtención, validación y configuración de los datos

---
### Cómo usar este cuaderno
1. Ejecute los pasos **en orden**, de arriba hacia abajo, con el botón ▶ que aparece a la izquierda de cada celda.
2. Cuando una celda haga una pregunta, escriba la respuesta en el recuadro y presione **Enter**. Si solo presiona Enter, se usa el valor sugerido entre corchetes.
3. No es necesario modificar el código. Las celdas marcadas con 🔧 contienen funciones y pueden permanecer cerradas.

> Las partes 3 (reglas de calidad) y 4 (reporte) se agregarán en las siguientes versiones.

## Paso 1. Preparar el entorno
Importa las bibliotecas que usa el prototipo. En Google Colab ya vienen instaladas.

In [ ]:
#@title ▶ Paso 1. Preparar el entorno
import importlib, subprocess, sys

for biblioteca in ["pandas", "requests"]:
    try:
        importlib.import_module(biblioteca)
    except ImportError:
        print(f"Instalando {biblioteca}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", biblioteca], check=True)

import pandas as pd
import requests
print(f"✅ Entorno listo · Python {sys.version.split()[0]} · pandas {pd.__version__}")

## Paso 2. Conectar Google Drive
Conecta su Google Drive para guardar la carpeta del proyecto, los parámetros y el registro de cada ejecución.
Cuando aparezca la ventana de permisos, elija su cuenta de Google y presione **Permitir**.

In [ ]:
#@title ▶ Paso 2. Conectar Google Drive
from pathlib import Path

NOMBRE_CARPETA = "EvaluadorCalidadDatosAbiertos"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_PROYECTO = Path("/content/drive/MyDrive") / NOMBRE_CARPETA
    CARPETA_TEMPORAL = Path("/content/temporal")
except ImportError:
    # Fuera de Colab (solo para pruebas): se trabaja en la carpeta actual.
    CARPETA_PROYECTO = Path.cwd() / NOMBRE_CARPETA
    CARPETA_TEMPORAL = Path.cwd() / "temporal"

CARPETA_PROYECTO.mkdir(parents=True, exist_ok=True)
CARPETA_TEMPORAL.mkdir(parents=True, exist_ok=True)
print(f"✅ Carpeta del proyecto: {CARPETA_PROYECTO}")

## Paso 3. Cargar las funciones
Carga las funciones de búsqueda, descarga, carga de archivos, parámetros y auditoría, y lee el archivo `parametros.json`.
Si es la primera vez, el archivo se crea con los valores por defecto.

In [ ]:
#@title 🔧 Funciones de la Parte 1 (no es necesario modificar esta celda) { display-mode: "form" }
# ---------------------------------------------------------------------------
# Capa de acceso a datos y gestor de ejecuciones (Fase 3, secciones 4.1 y 6.4)
#   - parametros: lectura y escritura de parametros.json (RF-11)
#   - ejecuciones y auditoria: carpeta por ejecución y registro de eventos
#   - busqueda: consulta del catálogo de datos.gov.co y descarga del CSV (RF-12)
#   - carga: archivo CSV propio desde el equipo del usuario (RF-01)
# ---------------------------------------------------------------------------
import csv
import hashlib
import io
import json
import re
import unicodedata
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

URL_CATALOGO = "https://www.datos.gov.co/api/catalog/v1"
DOMINIO_PORTAL = "www.datos.gov.co"
URL_RECURSO = "https://www.datos.gov.co/resource/{id}.csv"
URL_CONTEO = "https://www.datos.gov.co/resource/{id}.json"
URL_FICHA = "https://www.datos.gov.co/d/{id}"
PATRON_ID = re.compile(r"^[a-z0-9]{4}-[a-z0-9]{4}$")

PARAMETROS_POR_DEFECTO = {
    "version_parametros": "1.0",
    "marcadores_vacio": ["", "N/A", "NA", "NULL", "-"],
    "formatos_fecha": ["%Y-%m-%d", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S"],
    "max_resultados_busqueda": 20,
    "max_registros_descarga": 100000,
    "registros_por_pagina": 50000,
    "tamano_max_mb": 50,
    "max_ejemplos_por_regla": 10,
    "segundos_espera_red": 60,
}

ZONA_HORARIA = ZoneInfo("America/Bogota")  # Colab trabaja en hora UTC; se usa la hora de Colombia

SUBCARPETAS = ["01_configuracion", "02_busqueda", "03_fuente", "04_evaluacion", "05_reporte", "99_auditoria"]


# ----------------------------- utilidades ---------------------------------
def ahora():
    return datetime.now(ZONA_HORARIA).strftime("%Y-%m-%d %H:%M:%S")


def guardar_json(ruta, contenido):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    temporal = ruta.with_suffix(ruta.suffix + ".tmp")
    temporal.write_text(json.dumps(contenido, ensure_ascii=False, indent=2), encoding="utf-8")
    temporal.replace(ruta)


def leer_json(ruta, valor_si_no_existe=None):
    ruta = Path(ruta)
    if not ruta.exists():
        return valor_si_no_existe
    return json.loads(ruta.read_text(encoding="utf-8"))


def texto_para_carpeta(texto, largo=30):
    sin_tildes = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    limpio = re.sub(r"[^a-zA-Z0-9]+", "_", sin_tildes).strip("_").lower()
    return (limpio or "evaluacion")[:largo]


def preguntar(mensaje, valor_sugerido=""):
    """Pregunta al usuario. Si solo presiona Enter, se usa el valor sugerido."""
    ayuda = f" [Enter = {valor_sugerido}]" if valor_sugerido != "" else ""
    try:
        respuesta = input(f"{mensaje}{ayuda}: ").strip()
    except EOFError:
        respuesta = ""
    return respuesta or str(valor_sugerido)


def huella_sha256(ruta):
    calculadora = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            calculadora.update(bloque)
    return calculadora.hexdigest()


# ----------------------------- parámetros ---------------------------------
def cargar_parametros(carpeta_proyecto):
    """Lee parametros.json; si falta o tiene valores inválidos, usa los valores por defecto."""
    ruta = Path(carpeta_proyecto) / "parametros.json"
    guardados = leer_json(ruta, {}) or {}
    parametros = dict(PARAMETROS_POR_DEFECTO)
    for clave, valor in guardados.items():
        if clave not in PARAMETROS_POR_DEFECTO:
            continue
        esperado = type(PARAMETROS_POR_DEFECTO[clave])
        if isinstance(valor, esperado) and not (isinstance(valor, int) and valor <= 0):
            parametros[clave] = valor
        else:
            print(f"⚠️ El parámetro «{clave}» no es válido; se usa el valor por defecto.")
    if not ruta.exists():
        guardar_json(ruta, parametros)
        print("Se creó parametros.json con los valores por defecto.")
    return parametros


# ------------------------ ejecuciones y auditoría -------------------------
def crear_ejecucion(carpeta_proyecto, nombre):
    marca = datetime.now(ZONA_HORARIA).strftime("%Y%m%d-%H%M%S")
    identificador = f"{marca}_{texto_para_carpeta(nombre)}"
    carpeta = Path(carpeta_proyecto) / "ejecuciones" / identificador
    for sub in SUBCARPETAS:
        (carpeta / sub).mkdir(parents=True, exist_ok=True)
    ejecucion = {
        "id_ejecucion": identificador,
        "carpeta": str(carpeta),
        "fecha_hora": ahora(),
        "nombre": nombre,
        "origen": None,
        "estado": "creada",
    }
    guardar_json(carpeta / "01_configuracion" / "configuracion.json", ejecucion)
    guardar_json(carpeta / "99_auditoria" / "auditoria.json", [])
    registrar_evento(ejecucion, "crear_ejecucion", "ok", f"Carpeta {identificador}")
    return ejecucion


def actualizar_ejecucion(ejecucion, **cambios):
    ejecucion.update(cambios)
    guardar_json(Path(ejecucion["carpeta"]) / "01_configuracion" / "configuracion.json", ejecucion)


def registrar_evento(ejecucion, paso, resultado, detalle=""):
    ruta = Path(ejecucion["carpeta"]) / "99_auditoria" / "auditoria.json"
    eventos = leer_json(ruta, []) or []
    eventos.append({"momento": ahora(), "paso": paso, "resultado": resultado, "detalle": str(detalle)[:500]})
    guardar_json(ruta, eventos)


# ------------------------ búsqueda en datos.gov.co ------------------------
def buscar_conjuntos(tema, parametros):
    """Consulta el catálogo de datos.gov.co y devuelve una tabla con los conjuntos encontrados."""
    respuesta = requests.get(
        URL_CATALOGO,
        params={
            "q": tema,
            "domains": DOMINIO_PORTAL,
            "search_context": DOMINIO_PORTAL,
            "only": "dataset",
            "limit": parametros["max_resultados_busqueda"],
        },
        timeout=parametros["segundos_espera_red"],
    )
    respuesta.raise_for_status()
    filas = []
    for item in respuesta.json().get("results", []):
        recurso = item.get("resource", {})
        identificador = recurso.get("id", "")
        if recurso.get("type") != "dataset" or not PATRON_ID.match(identificador):
            continue
        filas.append({
            "id_conjunto": identificador,
            "titulo": recurso.get("name", ""),
            "entidad": recurso.get("attribution") or "Sin entidad registrada",
            "actualizado": (recurso.get("data_updated_at") or recurso.get("updatedAt") or "")[:10],
            "url_origen": URL_FICHA.format(id=identificador),
        })
    tabla = pd.DataFrame(filas, columns=["id_conjunto", "titulo", "entidad", "actualizado", "url_origen"])
    tabla.index = range(1, len(tabla) + 1)
    return tabla


def contar_registros(id_conjunto, parametros):
    try:
        respuesta = requests.get(URL_CONTEO.format(id=id_conjunto), params={"$select": "count(*) as total"},
                                 timeout=parametros["segundos_espera_red"])
        respuesta.raise_for_status()
        return int(respuesta.json()[0]["total"])
    except Exception:
        return None  # el conteo es informativo; si falla, la descarga continúa


def descargar_conjunto(id_conjunto, destino, parametros):
    """Descarga el conjunto por páginas hasta el máximo de registros y lo guarda como un único CSV."""
    total = contar_registros(id_conjunto, parametros)
    limite = parametros["max_registros_descarga"] if total is None else min(total, parametros["max_registros_descarga"])
    tamano_max = parametros["tamano_max_mb"] * 1024 * 1024
    encabezado, escritos, bytes_escritos = None, 0, 0
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    with destino.open("w", encoding="utf-8", newline="") as salida:
        escritor = csv.writer(salida)
        while escritos < limite:
            cantidad = min(parametros["registros_por_pagina"], limite - escritos)
            respuesta = requests.get(URL_RECURSO.format(id=id_conjunto),
                                     params={"$limit": cantidad, "$offset": escritos, "$order": ":id"},
                                     timeout=parametros["segundos_espera_red"])
            respuesta.raise_for_status()
            bytes_escritos += len(respuesta.content)
            if bytes_escritos > tamano_max:
                raise ValueError(f"El conjunto supera el tamaño máximo de {parametros['tamano_max_mb']} MB.")
            lector = csv.reader(io.StringIO(respuesta.content.decode("utf-8-sig"), newline=""))
            columnas = next(lector, None)
            registros = list(lector)
            if columnas is None or not registros:
                break
            if encabezado is None:
                encabezado = columnas
                escritor.writerow(encabezado)
            elif columnas != encabezado:
                raise ValueError("Las columnas cambiaron entre páginas; la descarga se detuvo.")
            escritor.writerows(registros)
            escritos += len(registros)
            print(f"   {escritos:,} registros descargados" + (f" de {limite:,}" if limite else ""))
            if len(registros) < cantidad:
                break
    if not escritos:
        destino.unlink(missing_ok=True)
        raise ValueError("El conjunto de datos no tiene registros para descargar.")
    cobertura = "completa" if total is not None and escritos >= total else (
        "parcial por límite" if total is not None else "sin conteo del portal")
    return {"registros_descargados": escritos, "registros_en_portal": total, "cobertura": cobertura}


# --------------------------- carga de archivo propio ----------------------
def cargar_archivo_propio(carpeta_temporal, parametros):
    """Pide al usuario un CSV desde su equipo (en Colab abre el selector de archivos)."""
    carpeta_temporal = Path(carpeta_temporal)
    carpeta_temporal.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files
    except ImportError:
        ruta = Path(preguntar("Escriba la ruta del archivo CSV"))
        if not ruta.exists():
            raise FileNotFoundError(f"No existe el archivo {ruta}")
        nombre, contenido = ruta.name, ruta.read_bytes()
    else:
        print("Seleccione el archivo CSV en la ventana que se abre...")
        subidos = files.upload()
        if not subidos:
            raise ValueError("No se seleccionó ningún archivo.")
        nombre, contenido = next(iter(subidos.items()))
    if not nombre.lower().endswith(".csv"):
        raise ValueError("El archivo debe tener extensión .csv.")
    if len(contenido) > parametros["tamano_max_mb"] * 1024 * 1024:
        raise ValueError(f"El archivo supera el tamaño máximo de {parametros['tamano_max_mb']} MB.")
    destino = carpeta_temporal / texto_para_carpeta(Path(nombre).stem, 60)
    destino = destino.with_suffix(".csv")
    destino.write_bytes(contenido)
    return destino, nombre


print("✅ Funciones de la Parte 1 cargadas.")


parametros = cargar_parametros(CARPETA_PROYECTO)
print("Parámetros en uso:")
for clave, valor in parametros.items():
    print(f"   {clave}: {valor}")

## Paso 4. Crear la ejecución
Cada uso del prototipo es una **ejecución**. Se crea una carpeta con la fecha, la hora y el nombre que usted escriba, donde quedarán la configuración, el reporte y la auditoría.

In [ ]:
#@title ▶ Paso 4. Crear la ejecución
nombre_ejecucion = preguntar("Escriba un nombre corto para esta ejecución (por ejemplo, el tema)", "evaluacion")
ejecucion = crear_ejecucion(CARPETA_PROYECTO, nombre_ejecucion)
print(f"✅ Ejecución creada: {ejecucion['id_ejecucion']}")
print(f"   Carpeta: {ejecucion['carpeta']}")

## Paso 5A. Buscar o cargar los datos
Elija de dónde vienen los datos:
- **1. Buscar en datos.gov.co:** escribe un tema y el prototipo muestra una lista numerada de los conjuntos de datos encontrados. La elección se hace en el **Paso 5B**.
- **2. Cargar un archivo propio:** selecciona un archivo CSV de su computador. En este caso el Paso 5B no es necesario.

El archivo de datos se guarda solo en el almacenamiento temporal de Colab; en Drive quedan únicamente los datos de la fuente y la auditoría.

In [ ]:
#@title ▶ Paso 5A. Buscar o cargar los datos
from pathlib import Path

ruta_datos = None
fuente = None
resultados_busqueda = None
tema_busqueda = None

def mostrar_vista(ruta, datos_fuente):
    guardar_json(Path(ejecucion["carpeta"]) / "03_fuente" / "fuente.json", datos_fuente)
    vista = pd.read_csv(ruta, dtype=str, nrows=5, sep=None, engine="python")
    print(f"\n✅ Datos listos para la validación ({datos_fuente['origen']}).")
    print("Primeros registros (la validación completa se hace en la Parte 2):")
    display(vista)

def obtener_desde_equipo():
    destino, nombre_original = cargar_archivo_propio(CARPETA_TEMPORAL, parametros)
    datos_fuente = {"origen": "archivo propio", "nombre_original": nombre_original,
                    "archivo_temporal": str(destino), "huella_sha256": huella_sha256(destino),
                    "tamano_bytes": destino.stat().st_size, "fecha_carga": ahora()}
    actualizar_ejecucion(ejecucion, origen="archivo propio", estado="datos obtenidos")
    registrar_evento(ejecucion, "carga", "ok", nombre_original)
    return destino, datos_fuente

def mostrar_lista(tabla):
    print(f"\nSe encontraron {len(tabla)} conjuntos de datos:\n")
    for numero, fila in tabla.iterrows():
        titulo = fila["titulo"] if len(fila["titulo"]) <= 75 else fila["titulo"][:72] + "..."
        print(f"{numero:>3}. {titulo}")
        print(f"      {fila['entidad']} · actualizado {fila['actualizado'] or 'sin fecha'}")

opcion = preguntar("¿De dónde vienen los datos? 1 = buscar en datos.gov.co, 2 = cargar un archivo propio", 1)
try:
    if opcion == "2":
        ruta_datos, fuente = obtener_desde_equipo()
        mostrar_vista(ruta_datos, fuente)
        print("\nComo cargó un archivo propio, no necesita el Paso 5B.")
    else:
        tema_busqueda = preguntar("¿Qué tema desea buscar en datos.gov.co?", ejecucion["nombre"])
        print(f"Buscando «{tema_busqueda}» en datos.gov.co...")
        resultados_busqueda = buscar_conjuntos(tema_busqueda, parametros)
        guardar_json(Path(ejecucion["carpeta"]) / "02_busqueda" / "resultados.json",
                     {"tema": tema_busqueda, "fecha_hora": ahora(),
                      "resultados": resultados_busqueda.to_dict(orient="records")})
        registrar_evento(ejecucion, "busqueda", "ok", f"Tema «{tema_busqueda}»: {len(resultados_busqueda)} conjuntos")
        if resultados_busqueda.empty:
            print("No se encontraron conjuntos de datos con ese tema. Ejecute de nuevo este paso con otro tema.")
        else:
            mostrar_lista(resultados_busqueda)
            print("\n👉 Revise la lista y ejecute el Paso 5B para escribir el número del conjunto que desea evaluar.")
except Exception as error:
    registrar_evento(ejecucion, "obtener_datos", "error", error)
    print(f"❌ No se pudieron obtener los datos: {error}")
    if opcion != "2":
        print("Puede intentar con otro tema o elegir la opción 2 para cargar un archivo propio.")

## Paso 5B. Elegir y descargar el conjunto
Escriba el número del conjunto de datos de la lista del Paso 5A. El prototipo lo descarga desde datos.gov.co y muestra los primeros registros.

In [ ]:
#@title ▶ Paso 5B. Elegir y descargar el conjunto
def pedir_numero(cantidad, intentos=5):
    for _ in range(intentos):
        respuesta = preguntar(f"Escriba el número del conjunto que desea evaluar (de 1 a {cantidad})")
        if respuesta.isdigit() and 1 <= int(respuesta) <= cantidad:
            return int(respuesta)
        print(f"   «{respuesta}» no es un número de la lista. Intente de nuevo.")
    raise ValueError("No se eligió un número válido.")

if fuente is not None:
    print("Los datos ya se obtuvieron en el Paso 5A. Puede continuar con la Parte 2.")
elif resultados_busqueda is None or resultados_busqueda.empty:
    print("Primero ejecute el Paso 5A y busque un tema que tenga resultados.")
else:
    try:
        numero = pedir_numero(len(resultados_busqueda))
        elegido = resultados_busqueda.loc[numero].to_dict()
        print(f"Descargando «{elegido['titulo']}»...")
        destino = CARPETA_TEMPORAL / f"{elegido['id_conjunto']}.csv"
        resumen = descargar_conjunto(elegido["id_conjunto"], destino, parametros)
        fuente = {**elegido, **resumen, "origen": "datos.gov.co", "tema_busqueda": tema_busqueda,
                  "archivo_temporal": str(destino), "huella_sha256": huella_sha256(destino),
                  "fecha_descarga": ahora()}
        ruta_datos = destino
        actualizar_ejecucion(ejecucion, origen="datos.gov.co", tema_busqueda=tema_busqueda,
                             id_conjunto=elegido["id_conjunto"], estado="datos obtenidos")
        registrar_evento(ejecucion, "descarga", "ok",
                         f"{elegido['id_conjunto']}: {resumen['registros_descargados']} registros ({resumen['cobertura']})")
        if resumen["cobertura"] == "parcial por límite":
            print(f"⚠️ El portal tiene {resumen['registros_en_portal']:,} registros; se descargaron los primeros "
                  f"{resumen['registros_descargados']:,} por el límite de los parámetros.")
        mostrar_vista(ruta_datos, fuente)
    except Exception as error:
        registrar_evento(ejecucion, "descarga", "error", error)
        print(f"❌ No se pudo descargar el conjunto: {error}")
        print("Puede ejecutar de nuevo este paso y elegir otro número.")

---
# Parte 2. Validación y configuración
En esta parte el prototipo revisa que el archivo tenga una estructura correcta y usted indica cómo se debe evaluar.

## Paso 6. Validar la estructura del archivo
El prototipo detecta la **codificación** y el **delimitador**, comprueba que todos los registros tengan el mismo número de campos que el encabezado y advierte si alguna columna parece contener **datos personales** (Ley 1581 de 2012). Si la estructura no es correcta, indica las líneas con problemas y no continúa.

In [ ]:
#@title ▶ Paso 6. Validar la estructura del archivo { display-mode: "form" }
# ---------------------------------------------------------------------------
# Capa de lógica de negocio (Fase 3, sección 4.1)
#   - validacion: delimitador, codificación, número de campos y aviso de datos personales (RF-02, RF-03, RNF-06)
#   - configuracion: columnas obligatorias, identificadoras y de fecha (RF-04)
# ---------------------------------------------------------------------------
import csv
import io
import re
import unicodedata
from pathlib import Path

import pandas as pd

CODIFICACIONES = [("utf-8-sig", "UTF-8"), ("cp1252", "Windows-1252"), ("latin-1", "ISO-8859-1")]
DELIMITADORES = {",": "coma (,)", ";": "punto y coma (;)", "\t": "tabulador", "|": "barra vertical (|)"}

# Palabras que, en el nombre de una columna, sugieren datos de personas naturales (Ley 1581 de 2012).
# Se comparan sin tildes; datos.gov.co además elimina las letras con tilde (c_dula, tel_fono).
PISTAS_PERSONALES = ["cedula", "c_dula", "documento", "apellido", "correo", "email", "e_mail",
                     "telefono", "tel_fono", "celular", "direccion", "direcci_n", "nacimiento",
                     "edad", "sexo", "genero", "g_nero", "pasaporte"]
PATRON_CORREO = re.compile(r"^[^@\s]+@[^@\s]+\.[a-zA-Z]{2,}$")
PATRON_CELULAR = re.compile(r"^(\+?57)?\s?3\d{2}\s?\d{3}\s?\d{4}$")

PATRONES_FECHA = [
    re.compile(r"^\d{4}-\d{2}-\d{2}([T ]\d{2}:\d{2}(:\d{2}(\.\d+)?)?)?$"),
    re.compile(r"^\d{1,2}/\d{1,2}/\d{4}( \d{1,2}:\d{2}(:\d{2})?)?$"),
    re.compile(r"^\d{1,2}-\d{1,2}-\d{4}$"),
    re.compile(r"^\d{4}/\d{2}/\d{2}$"),
]


def sin_tildes(texto):
    return unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii").lower()


# ------------------------------ validación --------------------------------
def detectar_codificacion(contenido):
    if b"\x00" in contenido[:65536]:
        raise ValueError("El archivo parece binario (contiene bytes nulos); no es un CSV de texto.")
    for codigo, nombre in CODIFICACIONES:
        try:
            return contenido.decode(codigo), codigo, nombre
        except UnicodeDecodeError:
            continue
    raise ValueError("No se reconoció la codificación del archivo.")


def detectar_delimitador(texto):
    muestra = texto[:65536]
    try:
        dialecto = csv.Sniffer().sniff(muestra, delimiters="".join(DELIMITADORES))
        return dialecto.delimiter
    except csv.Error:
        # Sniffer no pudo decidir: se elige el delimitador más frecuente en la primera línea.
        primera = muestra.splitlines()[0] if muestra else ""
        conteos = {d: primera.count(d) for d in DELIMITADORES}
        mejor = max(conteos, key=conteos.get)
        return mejor if conteos[mejor] > 0 else ","


def revisar_campos(texto, delimitador, max_ejemplos):
    """Compara el número de campos de cada registro con el del encabezado."""
    lector = csv.reader(io.StringIO(texto, newline=""), delimiter=delimitador)
    encabezado = next(lector, None)
    if not encabezado:
        raise ValueError("El archivo está vacío o no tiene encabezado.")
    esperados = len(encabezado)
    problemas, total, vacias = [], 0, 0
    for fila in lector:
        if not fila or all(c.strip() == "" for c in fila) and len(fila) <= 1:
            vacias += 1
            continue
        total += 1
        if len(fila) != esperados:
            problemas.append({"linea": lector.line_num, "campos": len(fila)})
    return encabezado, total, vacias, problemas, esperados


def revisar_encabezado(encabezado):
    """Detecta nombres de columna vacíos o repetidos y propone nombres utilizables."""
    avisos, nombres, vistos = [], [], {}
    for posicion, nombre in enumerate(encabezado, start=1):
        limpio = nombre.strip()
        if not limpio:
            limpio = f"columna_{posicion}"
            avisos.append(f"La columna {posicion} no tiene nombre; se llamará «{limpio}».")
        if limpio in vistos:
            vistos[limpio] += 1
            nuevo = f"{limpio}_{vistos[limpio]}"
            avisos.append(f"El nombre «{limpio}» está repetido; la columna {posicion} se llamará «{nuevo}».")
            limpio = nuevo
        else:
            vistos[limpio] = 1
        nombres.append(limpio)
    return nombres, avisos


def columnas_personales(datos):
    """Devuelve las columnas que podrían contener datos de personas naturales, con el motivo."""
    sospechosas = []
    for columna in datos.columns:
        nombre = sin_tildes(columna)
        pista = next((p for p in PISTAS_PERSONALES if p in nombre), None)
        if pista:
            sospechosas.append({"columna": columna, "motivo": f"el nombre contiene «{pista}»"})
            continue
        valores = datos[columna].str.strip()
        valores = valores[valores != ""].head(500)
        if len(valores) and valores.str.match(PATRON_CORREO).mean() > 0.5:
            sospechosas.append({"columna": columna, "motivo": "los valores parecen correos electrónicos"})
        elif len(valores) and valores.str.match(PATRON_CELULAR).mean() > 0.5:
            sospechosas.append({"columna": columna, "motivo": "los valores parecen números de celular"})
    return sospechosas


def validar_estructura(ruta, parametros):
    """Valida el CSV y, si la estructura es correcta, devuelve el DataFrame y un resumen."""
    contenido = Path(ruta).read_bytes()
    texto, codigo, nombre_codificacion = detectar_codificacion(contenido)
    delimitador = detectar_delimitador(texto)
    encabezado, total, vacias, problemas, esperados = revisar_campos(
        texto, delimitador, parametros["max_ejemplos_por_regla"])
    resumen = {
        "codificacion": nombre_codificacion,
        "delimitador": DELIMITADORES.get(delimitador, repr(delimitador)),
        "registros": total,
        "columnas": esperados,
        "lineas_vacias_omitidas": vacias,
        "registros_con_campos_distintos": len(problemas),
        "ejemplos_problemas": problemas[: parametros["max_ejemplos_por_regla"]],
        "estructura_valida": not problemas and total > 0,
    }
    if total == 0:
        resumen["motivo"] = "El archivo tiene encabezado pero ningún registro."
        return None, resumen
    if problemas:
        resumen["motivo"] = (f"{len(problemas)} registros no tienen {esperados} campos como el encabezado.")
        return None, resumen
    nombres, avisos = revisar_encabezado(encabezado)
    datos = pd.read_csv(io.StringIO(texto), sep=delimitador, dtype=str, keep_default_na=False,
                        skip_blank_lines=True, header=0, names=nombres, engine="python")
    resumen["avisos_encabezado"] = avisos
    resumen["nombres_columnas"] = nombres
    resumen["posibles_datos_personales"] = columnas_personales(datos)
    if esperados == 1:
        resumen["avisos_encabezado"].append(
            "El archivo tiene una sola columna; revise si el delimitador es el correcto.")
    return datos, resumen


# ---------------------------- configuración -------------------------------
def parece_fecha(serie):
    valores = serie.str.strip()
    valores = valores[valores != ""].head(300)
    if valores.empty:
        return False
    coincide = valores.apply(lambda v: any(p.match(v) for p in PATRONES_FECHA))
    return coincide.mean() >= 0.8


def sugerir_fechas(datos):
    sugeridas = []
    for posicion, columna in enumerate(datos.columns, start=1):
        nombre = sin_tildes(columna)
        if "fecha" in nombre or parece_fecha(datos[columna]):
            sugeridas.append(posicion)
    return sugeridas


def candidatas_identificador(datos):
    """Columnas sin vacíos y con todos sus valores distintos: pistas para elegir el identificador."""
    candidatas = []
    for posicion, columna in enumerate(datos.columns, start=1):
        valores = datos[columna].str.strip()
        if (valores != "").all() and valores.nunique() == len(valores) and len(valores) > 1:
            candidatas.append(posicion)
    return candidatas


def tabla_columnas(datos):
    filas = []
    for posicion, columna in enumerate(datos.columns, start=1):
        valores = datos[columna].str.strip()
        llenos = valores[valores != ""]
        ejemplo = llenos.iloc[0] if len(llenos) else ""
        filas.append({"n": posicion, "columna": columna,
                      "ejemplo": ejemplo if len(ejemplo) <= 40 else ejemplo[:37] + "...",
                      "valores_distintos": llenos.nunique()})
    return pd.DataFrame(filas).set_index("n")


def leer_numeros(texto, cantidad):
    """Convierte «1, 3, 5-7» en [1, 3, 5, 6, 7]. «0» significa ninguna columna."""
    texto = texto.replace(" ", "")
    if texto == "0":
        return []
    numeros = []
    for parte in texto.split(","):
        if not parte:
            continue
        if re.fullmatch(r"\d+-\d+", parte):
            inicio, fin = map(int, parte.split("-"))
            if inicio > fin:
                raise ValueError(f"El rango «{parte}» está al revés.")
            numeros.extend(range(inicio, fin + 1))
        elif parte.isdigit():
            numeros.append(int(parte))
        else:
            raise ValueError(f"«{parte}» no es un número.")
    fuera = [n for n in numeros if not 1 <= n <= cantidad]
    if fuera:
        raise ValueError(f"Los números {fuera} no están en la lista (de 1 a {cantidad}).")
    return sorted(set(numeros))


def pedir_columnas(mensaje, cantidad, sugeridas, texto_sugerido, intentos=5):
    for _ in range(intentos):
        respuesta = preguntar(mensaje, texto_sugerido)
        if respuesta == texto_sugerido:
            return list(sugeridas)
        try:
            return leer_numeros(respuesta, cantidad)
        except ValueError as error:
            print(f"   {error} Intente de nuevo.")
    raise ValueError("No se indicaron columnas válidas.")


def describir(numeros, columnas):
    return ", ".join(f"{n}. {columnas[n - 1]}" for n in numeros) if numeros else "ninguna"


print("✅ Funciones de la Parte 2 cargadas.")


def validar_y_mostrar():
    global datos, estructura
    datos, estructura = None, None
    if ruta_datos is None:
        print("Primero obtenga los datos en el Paso 5A (y el 5B si buscó en datos.gov.co).")
        return
    try:
        datos, estructura = validar_estructura(ruta_datos, parametros)
    except Exception as error:
        registrar_evento(ejecucion, "validacion", "error", error)
        actualizar_ejecucion(ejecucion, estado="estructura no válida")
        print(f"❌ No se pudo validar el archivo: {error}")
        return
    guardar_json(Path(ejecucion["carpeta"]) / "03_fuente" / "estructura.json", estructura)
    print(f"Codificación: {estructura['codificacion']}")
    print(f"Delimitador:  {estructura['delimitador']}")
    print(f"Registros:    {estructura['registros']:,}")
    print(f"Columnas:     {estructura['columnas']}")
    if estructura["lineas_vacias_omitidas"]:
        print(f"Líneas vacías omitidas: {estructura['lineas_vacias_omitidas']}")
    if datos is None:
        registrar_evento(ejecucion, "validacion", "error", estructura["motivo"])
        actualizar_ejecucion(ejecucion, estado="estructura no válida")
        print(f"\n❌ La estructura no es válida: {estructura['motivo']}")
        for problema in estructura.get("ejemplos_problemas", []):
            print(f"   Línea {problema['linea']}: tiene {problema['campos']} campos")
        print("Corrija el archivo o elija otro conjunto de datos y vuelva al Paso 5A.")
        return
    for aviso in estructura["avisos_encabezado"]:
        print(f"⚠️ {aviso}")
    if estructura["posibles_datos_personales"]:
        print("\n⚠️ Posibles datos personales. Revise si estas columnas tienen información de personas naturales;")
        print("   el prototipo no las guarda, pero el reporte puede mostrar ejemplos de sus valores:")
        for item in estructura["posibles_datos_personales"]:
            print(f"   - {item['columna']}: {item['motivo']}")
    registrar_evento(ejecucion, "validacion", "ok",
                     f"{estructura['registros']} registros, {estructura['columnas']} columnas, "
                     f"{estructura['codificacion']}, {estructura['delimitador']}")
    actualizar_ejecucion(ejecucion, estado="estructura válida")
    print("\n✅ La estructura del archivo es válida. Continúe con el Paso 7.")

validar_y_mostrar()

## Paso 7. Configurar la evaluación
El prototipo muestra las columnas numeradas y hace tres preguntas. Responda con los **números** de las columnas separados por comas (por ejemplo `1, 3, 5`) o con un rango (por ejemplo `2-6`). Escriba `0` si no quiere elegir ninguna.
- **Columnas obligatorias:** las que siempre deben tener un valor. Con Enter se toman todas.
- **Columnas identificadoras:** las que no deben repetirse, como un código. Con Enter, ninguna; entonces la unicidad se evalúa solo sobre registros completos.
- **Columnas de fecha:** las que deben tener fechas en un mismo formato. Con Enter se toman las que el prototipo sugiere.

In [ ]:
#@title ▶ Paso 7. Configurar la evaluación
def configurar():
    global configuracion
    configuracion = None
    if datos is None:
        print("Primero ejecute el Paso 6 con un archivo de estructura válida.")
        return
    columnas = list(datos.columns)
    cantidad = len(columnas)
    print("Columnas del archivo:")
    for n, fila in tabla_columnas(datos).iterrows():
        print(f"{n:>4}. {fila['columna']}")
        print(f"        ejemplo: {fila['ejemplo'] or '(vacía)'} · {fila['valores_distintos']:,} valores distintos")

    obligatorias = pedir_columnas("\n1) ¿Qué columnas son obligatorias?", cantidad, range(1, cantidad + 1), "todas")
    pistas = candidatas_identificador(datos)
    if pistas:
        print(f"   Pista: estas columnas no tienen vacíos ni valores repetidos: {describir(pistas, columnas)}")
    identificadoras = pedir_columnas("2) ¿Qué columnas son identificadoras?", cantidad, [], "ninguna")
    sugeridas = sugerir_fechas(datos)
    texto_fechas = ", ".join(map(str, sugeridas)) if sugeridas else "ninguna"
    if sugeridas:
        print(f"   Sugeridas como fecha: {describir(sugeridas, columnas)}")
    fechas = pedir_columnas("3) ¿Qué columnas contienen fechas?", cantidad, sugeridas, texto_fechas)

    configuracion = {
        "fecha_hora": ahora(),
        "columnas": [{"posicion": n, "nombre": columna,
                      "es_obligatoria": n in obligatorias,
                      "es_identificador": n in identificadoras,
                      "es_fecha": n in fechas} for n, columna in enumerate(columnas, start=1)],
    }
    guardar_json(Path(ejecucion["carpeta"]) / "01_configuracion" / "configuracion_evaluacion.json", configuracion)
    registrar_evento(ejecucion, "configuracion", "ok",
                     f"{len(obligatorias)} obligatorias, {len(identificadoras)} identificadoras, {len(fechas)} de fecha")
    actualizar_ejecucion(ejecucion, estado="evaluación configurada")
    print("\nResumen de la configuración:")
    print(f"   Obligatorias ({len(obligatorias)}): " + ("todas" if len(obligatorias) == cantidad else describir(obligatorias, columnas)))
    print(f"   Identificadoras: {describir(identificadoras, columnas)}")
    print(f"   Fechas: {describir(fechas, columnas)}")
    print("\n✅ Evaluación configurada. Las reglas de calidad se aplican en la Parte 3.")

try:
    configurar()
except Exception as error:
    registrar_evento(ejecucion, "configuracion", "error", error)
    print(f"❌ No se pudo configurar la evaluación: {error}")
    print("Ejecute de nuevo este paso.")

---
### Fin de la Parte 2
Revise en Google Drive la carpeta de esta ejecución: en `03_fuente` debe estar `estructura.json` y en `01_configuracion` el archivo `configuracion_evaluacion.json`.

**Siguiente parte:** aplicación de las reglas de completitud, unicidad y consistencia (paso 8).